<img src="https://upload.wikimedia.org/wikipedia/commons/e/e7/Dalhouise_University_Watermark.png" alt="Dal logo" width="200"/>

# CSCI 3151 — Foundations of Machine Learning
# Assignment 2 — What Should a Classifier Ignore?

**Released:** 4 October 2026  
**Due:** 30 October 2026, 23h59 AT  
**Work:** one submission per assigned/approved team of **2 or 3** students

**Team members:**  
- Cameron Caissie / cm598262
- Kai Patel / kz685937
- Maxime Schuler / mx867755

---

A sensor classifier can fail for two very different reasons: it may not have enough capacity to express the decision rule, or it may be sensitive to variation that the task says should not matter. In this assignment you will investigate both.

You will work with a reproducible synthetic monitoring problem built from scikit-learn primitives. The system has four sensors. Two underlying physical contrasts determine the operating regime; a **common calibration offset** can move all four sensor readings together without changing that regime. This gives us a controlled setting in which to ask:

> **What should a classifier ignore, and how can representation, kernels, augmentation, and dimensionality reduction help or hurt?**

The point is not to chase the highest possible accuracy. Strong submissions make controlled comparisons, distinguish model capacity from representation choice, and support claims with evidence.


## Assessment

| # | Section | Mark | Out of | Comments |
|---|---|---:|---:|---|
| 0 | Reproducibility, team plan, and data contract |  | /2 | |
| 1 | Feature geometry, kernels, and margins |  | /10 | |
| 2 | Build an invariant representation |  | /7 | |
| 3 | Augmentation as a hypothesis |  | /8 | |
| 4 | PCA, variance, and visual evidence |  | /8 | |
| 5 | Frozen evaluation and evidence memo |  | /5 | |
|   | **TOTAL** |  | **/40** | |

### Rules and expectations

- Work only with your assigned/approved team. Every team member is responsible for understanding the complete submission.
- Unless a task explicitly says otherwise, use **training data to fit**, **validation data to choose**, and the **final set only after you freeze your choices in Part 5**.
- Any preprocessing learned from data must be fit on training data only. Pipelines are strongly encouraged.
- You may add cells, but do not delete required prompts, contracts, or check cells.
- Keep plots legible and label axes, legends, and conditions.
- Short written answers should normally be precise rather than long. A few strong sentences are better than a page of generic discussion.


## How the assignment maps to the modules

The work is organized around the three A2 lab modules:

- **M21 — kernel SVMs in practice:** Part 1
- **M25 — feature pipelines & augmentation:** Parts 2–3
- **M28 — visualizing high-dimensional data:** Part 4 and the Part 5 synthesis

Earlier material on evaluation, cross-validation, regularization, and leakage remains part of the expected workflow.


## The monitoring problem

Each observation comes from one machine snapshot. Four raw channels are recorded:

- `sensor_A`, `sensor_B`, `sensor_C`, `sensor_D`

The physical system has two latent **contrasts**, which can be estimated from paired sensors:

$$
d_1 = A-C, \qquad d_2 = B-D.
$$

The target is binary:

- `0`: the two contrasts are in **opposed** directions;
- `1`: the two contrasts are in **coupled** directions.

A calibration event may add approximately the same offset to **all four raw sensors at once**. According to the task definition, such a common-mode shift is a nuisance: it should **not** change the class. By contrast, flipping the sign of only one physical contrast changes a coupled regime to an opposed regime or vice versa, so that transformation is **not** label-preserving.

In this synthetic generator the common offset is exactly shared; independent sensor noise is added separately. Labels are assigned from the latent contrasts before that noise, so the observed differences are noisy estimates rather than an exact labelling rule.

You should treat these statements as the domain contract for this assignment. Your job is to test which modelling choices respect it.


## Part 0 — Reproducibility, team plan, and data contract  **/2**


In [1]:
# Part 0.1 — imports and environment
import sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.datasets import make_blobs
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

RANDOM_STATE = 3151

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)


Python: 3.14.7
NumPy: 2.5.3
pandas: 3.0.6
scikit-learn: 1.9.1


In [2]:
# Part 0.2 — supplied data generator. Do not modify.

def make_sensor_data(n, common_sd, seed, condition):
    """Create one split of the four-sensor XOR-style monitoring problem."""
    rng = np.random.default_rng(seed)
    latent, centre_id = make_blobs(
        n_samples=n,
        centers=[(-2, -2), (-2, 2), (2, -2), (2, 2)],
        cluster_std=0.80,
        random_state=seed,
    )
    u, v = latent[:, 0], latent[:, 1]
    y = ((u * v) > 0).astype(int)

    common = rng.normal(0.0, common_sd, size=n)
    eps = rng.normal(0.0, 0.35, size=(n, 4))

    A = common + u + eps[:, 0]
    B = common + v + eps[:, 1]
    C = common - u + eps[:, 2]
    D = common - v + eps[:, 3]

    return pd.DataFrame({
        "sensor_A": A,
        "sensor_B": B,
        "sensor_C": C,
        "sensor_D": D,
        "target": y,
        "condition": condition,
    })

train_df = make_sensor_data(900, common_sd=3.0, seed=3151, condition="nominal_train")
val_df = make_sensor_data(400, common_sd=3.0, seed=3152, condition="nominal_validation")
shift_val_df = make_sensor_data(400, common_sd=5.0, seed=3153, condition="shifted_validation")

# Final data are created now for reproducibility, but must not be inspected or used
# for fitting/tuning until Part 5, after you fill in the frozen record.
final_nominal = make_sensor_data(300, common_sd=3.0, seed=3154, condition="final_nominal")
final_shifted = make_sensor_data(300, common_sd=6.0, seed=3155, condition="final_shifted")
final_df = pd.concat([final_nominal, final_shifted], ignore_index=True)

FEATURES = ["sensor_A", "sensor_B", "sensor_C", "sensor_D"]

def split_xy(df):
    return df[FEATURES].to_numpy(float), df["target"].to_numpy(int)

X_train, y_train = split_xy(train_df)
X_val, y_val = split_xy(val_df)
X_shift_val, y_shift_val = split_xy(shift_val_df)

print(train_df.shape, val_df.shape, shift_val_df.shape)
print(train_df["target"].value_counts(normalize=True).sort_index())


(900, 6) (400, 6) (400, 6)
target
0    0.505556
1    0.494444
Name: proportion, dtype: float64


### Part 0.3 — team plan and contract check

Complete the table before the main experiments. “Lead” means first pass; “reviewer” means independently checks the reasoning/code/results. Every member remains responsible for all parts.

| Part | Lead | Reviewer |
|---|---|---|
| 1 | Maxime Schuler | Kai Patel |
| 2 | Kai Patel | Cameron Caissie |
| 3 | Cameron Caissie | Maxime Schuler |
| 4 | Maxime Schuler | Kai Patel |
| 5 | Kai Patel | Cameron Caissie |

Then answer briefly:

1. Why is adding the same constant to `A,B,C,D` supposed to preserve the label?

Adding the same constant to `A,B,C,D` preseves the label since they are defined by the two contrasts mentioned above. Adding a constant to these equations leads in the constants cancelling out. For example when adding 5 to each term of the first contrast yields: $d_1=(A+5)-(C+5)=A-C$. This shows that when adding a constant, the label is preserved.

2. Why would flipping the sign of **only one** physical contrast not preserve the label?

Since the label is defined by the contrasts' directions, changing the sign of one would mean either are now both facing the same direction or opposite directions (depending on their initial state). Thus, the label would flip when one of the contrasts' sign is flipped.

3. Confirm that you will not inspect `final_df` until the Part 5 frozen record is complete.

**Response:** We will not inspect `final_df` until we have frozen the record in Part 5.


## Part 1 — Feature geometry, kernels, and margins  **/10**


### Part 1.1 — establish baselines before engineering features  **/3**

Fit the following **leakage-free** models on the four raw sensors:

1. linear SVM, `SVC(kernel="linear", C=1)`;
2. RBF SVM over the grid `C in {0.1, 1, 10}` and `gamma in {0.03, 0.1, 0.3, 1}`.

Use `StandardScaler` inside a `Pipeline`. For every setting report:

- training balanced accuracy;
- nominal-validation balanced accuracy;
- shifted-validation balanced accuracy;
- number of support vectors (sum over classes).

Choose one RBF setting using the two validation sets and state your selection rule **before** using any final data. Your rule does not have to maximize one number; it must be clear and reproducible.


In [38]:
# Part 1.1
# Produce a tidy DataFrame with one row per model/setting.

C_values = [0.1, 1.0, 10.0]
gamma_values = [0.03, 0.1, 0.3, 1.0]

results = []

linear_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1))
])
linear_clf.fit(X_train, y_train)

y_train_pred = linear_clf.predict(X_train)
y_val_pred = linear_clf.predict(X_val)
y_shift_pred = linear_clf.predict(X_shift_val)

train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
val_bal_acc = balanced_accuracy_score(y_val, y_val_pred)
shift_bal_acc = balanced_accuracy_score(y_shift_val, y_shift_pred)

n_sv = linear_clf.named_steps["svm"].n_support_.sum()

results.append({
    "model": "Linear",
    "C": 1,
    "gamma": None,
    "train_bal_acc": train_bal_acc,
    "val_bal_acc": val_bal_acc,
    "shift_bal_acc": shift_bal_acc,
    "n_support_vectors": n_sv
})

for C in C_values:
    for gamma in gamma_values:
        rbf_clf = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="rbf", C=C, gamma=gamma))
        ])
        rbf_clf.fit(X_train, y_train)

        y_train_pred = rbf_clf.predict(X_train)
        y_val_pred = rbf_clf.predict(X_val)
        y_shift_pred = rbf_clf.predict(X_shift_val)
        
        train_bal_acc = balanced_accuracy_score(y_train, y_train_pred)
        val_bal_acc = balanced_accuracy_score(y_val, y_val_pred)
        shift_bal_acc = balanced_accuracy_score(y_shift_val, y_shift_pred)

        n_sv = rbf_clf.named_steps["svm"].n_support_.sum()

        results.append({
            "model": "RBF",
            "C": C,
            "gamma": gamma,
            "train_bal_acc": train_bal_acc,
            "val_bal_acc": val_bal_acc,
            "shift_bal_acc": shift_bal_acc,
            "n_support_vectors": n_sv
        })

results_11 = pd.DataFrame(results)
display(results_11)

rbf_results = results_11[results_11["model"] == "RBF"]
ranked_rbf = rbf_results.sort_values(
    by=["shift_bal_acc", "val_bal_acc"],
    ascending=False
)

selected_rbf = ranked_rbf.iloc[0]
print("Selected RBF:")
display(selected_rbf.to_frame().T)

,model,C,gamma,train_bal_acc,val_bal_acc,shift_bal_acc,n_support_vectors
0,Linear,1.0,NaN,0.505988,0.504638,0.504350,887
1,RBF,0.1,0.03,0.500000,0.500000,0.500000,892
2,RBF,0.1,0.10,0.943116,0.932311,0.832158,837
3,RBF,0.1,0.30,0.975478,0.964849,0.942319,545
4,RBF,0.1,1.00,0.991110,0.982412,0.965297,435
5,RBF,1.0,0.03,0.976528,0.969874,0.922417,770
6,RBF,1.0,0.10,0.988863,0.977387,0.952220,366
7,RBF,1.0,0.30,0.992283,0.984950,0.964846,208
8,RBF,1.0,1.00,0.991135,0.977437,0.967822,164
9,RBF,10.0,0.03,0.991085,0.982412,0.962471,320


Selected RBF:


,model,C,gamma,train_bal_acc,val_bal_acc,shift_bal_acc,n_support_vectors
8,RBF,1.0,1.0,0.991135,0.977437,0.967822,164


**Part 1.1 interpretation (about 4–7 sentences):**

- What evidence says that a linear boundary in the raw standardized feature space is or is not adequate?
- How do `C` and `gamma` change fit, robustness, and the support-vector count?
- State your RBF selection rule and selected setting.

**Response:**

The train_bal_acc, val_bal_acc, and shift_bal_acc of the linear model are all consistent at around 0.505, which is a rather low accuracy and therefore not adequate. The same values for the RBF model are generally much higher accuracy. As the C value grows, it tends to increase the accuracy and lower the number of support vectors. As the gamma value grows, it tends to increase the robustness and consitency and lower the number of support vectors. My selection rule is pick the C and gamme combination that gives the highest shift_bal_acc value, with val_bal_acc as a tie breaker. The selected setting is C = 1.0, gamma = 1.0.

### Part 1.2 — make one nonlinear feature map explicit  **/4**

Define the contrast vector

$$
z(x) = [A-C,\; B-D].
$$

Then implement

$$
\phi(z) = [z_1,\; z_2,\; z_1z_2,\; z_1^2,\; z_2^2].
$$

For a small sample of at least 8 training observations:

1. compute `Phi @ Phi.T` explicitly;
2. implement `explicit_kernel(Z1, Z2)` independently using vectorized coordinate products: do **not** call `phi_map`, construct its five-column feature matrices, or loop over observation pairs; return an `(n1, n2)` matrix for inputs of shapes `(n1, 2)` and `(n2, 2)`;
3. show numerically that the two Gram matrices agree to floating-point tolerance;
4. show that your Gram matrix is symmetric and report its smallest eigenvalue.

Use unscaled contrasts for this check. Tiny negative eigenvalues at floating-point roundoff scale are not a failure; assess them relative to the largest eigenvalue rather than requiring strict positivity.

You are not being asked to derive a new kernel family. You are verifying, concretely, that an explicit feature map induces a kernel through inner products.


In [39]:
# Part 1.2

def contrast_features(X):
    """Return [A-C, B-D] for an (n,4) raw sensor matrix."""
    return np.column_stack([X[:, 0] - X[:, 2], X[:, 1] - X[:, 3]])


def phi_map(Z):
    """Return [z1, z2, z1*z2, z1^2, z2^2]."""
    return np.column_stack([Z[:, 0], Z[:, 1], Z[:, 0] * Z[:, 1], Z[:, 0] ** 2, Z[:, 1] ** 2])


def explicit_kernel(Z1, Z2):
    """Return the (n1,n2) induced kernel using coordinate products, not phi_map."""
    return (Z1[:, 0:1] * Z2[:, 0]) + (Z1[:, 1:2] * Z2[:, 1]) + ((Z1[:, 0:1] * Z1[:, 1:2]) * (Z2[:, 0] * Z2[:, 1])) + ((Z1[:, 0:1] ** 2) * (Z2[:, 0] ** 2)) + ((Z1[:, 1:2] ** 2) * (Z2[:, 1] ** 2))

# Add your numerical verification below.

Z_train = contrast_features(X_train)
Phi = phi_map(Z_train[:8])
G_explicit = Phi @ Phi.T
G_kernel = explicit_kernel(Z_train[:8], Z_train[:8])

print("Gram matrices agree:", np.allclose(G_explicit, G_kernel))
print("Gram matrix shape:", G_kernel.shape)
print("Is symmetric:", np.allclose(G_kernel, G_kernel.T))

eigenvalues = np.linalg.eigvalsh(G_kernel)
smallest_eigenvalue = eigenvalues[0]
largest_eigenvalue = eigenvalues[-1]

print("Smallest eigenvalue:", smallest_eigenvalue)
print("Largest eigenvalue:", largest_eigenvalue)
print("Relative magnitude of smallest eigenvalue:", abs(smallest_eigenvalue) / largest_eigenvalue)

Gram matrices agree: True
Gram matrix shape: (8, 8)
Is symmetric: True
Smallest eigenvalue: -1.4686312768013675e-13
Largest eigenvalue: 8789.232081689692
Relative magnitude of smallest eigenvalue: 1.6709437902554846e-17


### Part 1.3 — is a better representation enough?  **/3**

Fit a **linear** SVM (`SVC(kernel="linear", C=1)`) on `phi(z(x))`. Compute contrasts and `phi` from the **unscaled raw readings first**, then fit `StandardScaler` on the training feature-map values before fitting the SVM. Do not standardize the four sensors separately before taking their differences. Compare it with:

- the raw-feature linear SVM;
- your selected raw-feature RBF SVM.

Report nominal and shifted validation balanced accuracy and support-vector count. Then answer:

1. Why can a linear classifier succeed after `phi` even though it failed on the raw sensors?
2. In this problem, what is one advantage and one disadvantage of an explicit feature map relative to the RBF kernel?
3. Does “nonlinear model” necessarily mean “nonlinear decision rule in the representation being used”? Explain.

**Response:**

The linear model originally scored about 50% accuracy, which suggests a fully linear boudary is simply inadequate in the original feature space. The feature map with phi adds squared terms that have more depth and allow the SVM to represent more complex nolinear relationships.

One advantage of an explicit feature map over the RBF kernel is that the model can be trained using a normal linear SVM. One disadvantage is that its performance explicitly depends on the chosen features, so it can fail to capture patterns not captured by the map.

No, a SVM with phi is linear in the transformed feature space but still produces a nonlinear decision boundary in the original space from the censors.

In [45]:
# Part 1.3 comparison table

Z_train = contrast_features(X_train)
Z_val = contrast_features(X_val)
Z_shift_val = contrast_features(X_shift_val)
Phi_train = phi_map(Z_train)
Phi_val = phi_map(Z_val)
Phi_shift_val = phi_map(Z_shift_val)

linear_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1))
])
linear_clf.fit(Phi_train, y_train)

y_val_pred = linear_clf.predict(Phi_val)
y_shift_pred = linear_clf.predict(Phi_shift_val)

val_bal_acc = balanced_accuracy_score(y_val, y_val_pred)
shift_bal_acc = balanced_accuracy_score(y_shift_val, y_shift_pred)

n_sv = linear_clf.named_steps["svm"].n_support_.sum()

feature_map_result = pd.DataFrame([{
    "model": "Linear + phi",
    "val_bal_acc": val_bal_acc,
    "shift_bal_acc": shift_bal_acc,
    "n_support_vectors": n_sv
}])

linear_result = results_11[results_11["model"] == "Linear"]
rbf_result = selected_rbf.to_frame().T

baseline_results = pd.concat([linear_result, rbf_result])
baseline_results = baseline_results[["model", "val_bal_acc", "shift_bal_acc", "n_support_vectors"]]

results_13 = pd.concat([baseline_results, feature_map_result], ignore_index=True)
results_13


,model,val_bal_acc,shift_bal_acc,n_support_vectors
0,Linear,0.504638,0.50435,887
1,RBF,0.977437,0.967822,164
2,Linear + phi,0.979899,0.99505,60


## Part 2 — Build an invariant representation  **/7**


The task contract says a common additive shift to all four sensors is irrelevant. A strong representation should make that fact easy for the classifier to respect.

### Part 2.1 — prove and test an invariance  **/3**

Implement `contrast_features` if you have not already done so. Then:

1. algebraically show what happens to `A-C` and `B-D` when the same constant `c` is added to all four sensors;
2. verify the result numerically for at least 20 random training rows and at least 10 different offsets spanning roughly `[-10, 10]`;
3. write an `assert` that would fail if your transform were not invariant to this common-mode shift.

**Derivation and interpretation:** TODO


In [ ]:
# TODO: Part 2.1 numerical invariance test


### Part 2.2 — compare representations under distribution shift  **/4**

Using the same model-selection discipline as before, compare at least these three systems:

- selected RBF SVM on raw sensors;
- RBF SVM on the 2D contrast representation;
- linear SVM on the explicit `phi(contrast)` representation.

For the contrast RBF model, compute contrasts from unscaled raw readings, then fit a `StandardScaler` on the training contrasts only. Tune only over the same small `C`/`gamma` grid from Part 1. Report train, nominal-validation, and shifted-validation balanced accuracy.

Then answer:

- Which change matters more here: changing the classifier or changing the representation?
- Does the shifted-validation result support the claim that the contrast representation encoded a useful invariance? Why?
- Identify one limitation of concluding “invariance solved robustness” from this experiment.

**Response:** TODO


In [ ]:
# TODO: Part 2.2 model comparison
results_22 = None
results_22


## Part 3 — Augmentation as a hypothesis  **/8**


Augmentation is not “free extra data.” It asserts that a transformation should preserve the target.

### Part 3.1 — implement a valid augmentation and a size-matched control  **/4**

Write `augment_common_shift(X, y, shift_sd, copies, seed)` that returns `(X_aug, y_aug)`, with the originals first followed by `copies` transformed blocks and their unchanged labels. Leave `X` and `y` unchanged. Use a local `np.random.default_rng(seed)` to draw an independent zero-mean Gaussian shift of standard deviation `shift_sd` for each row in each copy; add that scalar to all four raw channels. Use **training data only**, with `seed=3156` for the required experiment (distinct from the data-generation seeds).

Run a controlled comparison using the **selected raw-feature RBF setting from Part 1.1**, subject only to the sample-count correction below:

1. original training data;
2. duplicated training data with the same number of added rows as condition 3, but no transformation;
3. common-shift augmentation with `shift_sd=4.0`, one augmented copy per training row.

For **all conditions in Parts 3.1–3.2**, reuse the scaler fitted on the original `X_train`; transform each completed raw dataset with it and fit an `SVC` on those transformed values. Do not refit the scaler on augmented data. Keep the selected numerical `gamma` fixed.

**Control for sample count:** `SVC` penalizes a *sum* of per-row losses. Use the selected `C` for the original data and `C / (1 + copies)` when originals plus copies are used: here, `C / 2` for duplication, valid augmentation, and bad augmentation. This keeps the total loss weight comparable; it is not hyperparameter retuning.

Report nominal- and shifted-validation balanced accuracy. Do not retune the model separately for the conditions. An unchanged or worse score is a valid result; report what happened rather than searching for a required improvement.


In [ ]:
# TODO: Part 3.1

def augment_common_shift(X, y, shift_sd=4.0, copies=1, seed=3156):
    """Return original rows then shifted copies and labels; do not mutate inputs."""
    raise NotImplementedError

results_31 = None
results_31


**Interpretation:** Does the augmentation appear to help because there are more rows, or because of the transformation itself? What evidence distinguishes those explanations?

**Response:** TODO


### Part 3.2 — deliberately violate the label-preserving assumption  **/4**

Construct one **bad augmentation** that flips the sign of the first physical contrast while leaving the second unchanged, but incorrectly keeps the original label. One way to do this in raw sensor space is to swap `sensor_A` and `sensor_C` while leaving `B,D` alone.

1. Explain from the task definition why this transformation is not label-preserving.
2. Add one bad copy per training row and fit the same raw-feature RBF setting, frozen scaler, and `C / 2` sample-count correction used for the two-copy datasets in Part 3.1.
3. Compare nominal and shifted validation performance with the original and valid-augmentation conditions.
4. Give one general lesson about domain knowledge and augmentation that is supported by this experiment.

Do not spend time tuning the bad augmentation; it is a diagnostic, not a model candidate.


In [ ]:
# TODO: Part 3.2 bad augmentation and comparison
results_32 = None
results_32


**Response:** TODO


## Part 4 — PCA, variance, and visual evidence  **/8**


### Part 4.1 — PCA from first principles  **/4**

Use **only the raw, unstandardized training sensor matrix** for this part; do not whiten. Implement a small PCA routine using NumPy:

1. compute the training mean and center the data;
2. compute the sample covariance matrix using the denominator `n - 1`;
3. use `np.linalg.eigh`;
4. sort eigenpairs from largest to smallest eigenvalue;
5. return a dictionary with keys `mean`, `components`, `eigenvalues`, and `explained_variance_ratio`. Store unit component vectors as **rows**, ordered by decreasing eigenvalue.

Check your implementation against `PCA(svd_solver="full", whiten=False)` on exactly the same training matrix. Because component signs are arbitrary, compare quantities that are sign-safe: explained-variance ratios and absolute component-direction agreement.

Then report the smallest `k` that explains at least 90% of the raw training variance.


In [ ]:
# TODO: Part 4.1

def pca_numpy(X):
    """Return mean, row-wise components, eigenvalues, explained_variance_ratio."""
    raise NotImplementedError

# Compare against sklearn PCA here.


### Part 4.2 — when the highest-variance direction is mostly nuisance  **/2**

For each raw training row, compute the simple common-mode proxy

$$
m = (A+B+C+D)/4.
$$

Center the raw training data using the stored training mean and project it onto the first two components from Part 4.1. Report:

- correlation between PC1 score and `m`, interpreting its **magnitude** because component signs are arbitrary;
- class-conditional means of PC1 and PC2;
- the first-component loading vector.

Then answer: **why can a component explain a large fraction of variance while being relatively unhelpful for classification?** Tie your answer to the loadings, correlation, and task contract rather than giving only a generic PCA definition. Similar class-conditional means alone do **not** establish that a component contains no class information.


In [ ]:
# TODO: Part 4.2


**Response:** TODO


### Part 4.3 — PCA vs t-SNE: what may we actually claim?  **/2**

Use the **same 400 nominal-validation observations (`X_val`, `y_val`)** in all four plots. Fit raw-sensor PCA on `X_train` and contrast PCA on `contrast_features(X_train)`, then transform the corresponding validation inputs. Use unstandardized inputs and no whitening for these plots, as in Part 4.1.

1. a 2D PCA plot of the raw validation sensors;
2. a 2D PCA plot of the invariant contrast representation;
3. two t-SNE plots of the invariant contrast representation, each using `perplexity=30`, `init="random"`, and `learning_rate=200`, with `random_state=3151` and `3152`, respectively. Keep all other settings and row order identical.

Fit t-SNE directly to the validation contrasts for this **exploratory visualization only**; use labels only to colour points. This is an explicit exception to the fitting rule, not a predictive transform or evidence of held-out classifier performance. The contrast PCA is a 2D-to-2D coordinate-change baseline, not dimensionality reduction.

Use the same class colours and similar point sizes across all four plots.

Answer briefly:

- Which plot(s) make the classes look most separated?
- Which distances or cluster sizes are interpretable, and which are not?
- Does agreement between two t-SNE seeds prove that the visible clusters are “real”? Why or why not?
- Give one claim that would be **too strong** to make from these figures alone.


In [ ]:
# TODO: Part 4.3 visualizations


**Response:** TODO


## Part 5 — Frozen evaluation and evidence memo  **/5**


### Part 5.1 — freeze before opening the final set  **/2**

Before computing **any** final-set metric, complete this record using only Parts 1–4.

```text
FROZEN MODEL RECORD
representation: TODO
classifier: TODO
preprocessing pipeline: TODO
hyperparameters: TODO
augmentation (if any; include copies, shift_sd, seed, and C correction): TODO
refit on training + both validation sets: YES / NO
selection rule used: TODO
primary metric: balanced accuracy
secondary diagnostic(s): TODO
expected strength: TODO
expected failure mode / limitation: TODO
```

Your final model must be one you actually evaluated in the development work above. Decide and record **before final inspection** whether to retain that fitted model or refit it on **training + both validation sets**. If refitting, fit learned preprocessing on the unaugmented combined development rows, then apply the frozen augmentation recipe and the Part 3 sample-count correction, if relevant. This is the permitted supervised-refitting exception after freezing. You may not change the refit decision, representation, augmentation rule, model family, or hyperparameters after seeing final results.

**Frozen record completed before final inspection:** YES / NO


### Part 5.2 — one final evaluation  **/1**

Now, and only now, evaluate the frozen pipeline on `final_df`.

Report balanced accuracy and binary F1, treating `target=1` as the positive class:

- overall;
- `final_nominal` only;
- `final_shifted` only.

Also show the overall confusion matrix. Do not run a second round of model selection after seeing these results.


In [ ]:
# TODO: Part 5.2 — final evaluation only after the frozen record above is complete
final_results = None
final_results


### Part 5.3 — evidence memo  **/2**

Write **350–500 words** answering the question:

> **What should this classifier ignore, and what did your experiments show about the best way to make it ignore that variation?**

Your memo must:

- make one clear recommendation;
- cite at least three concrete numerical results from different parts of the assignment;
- distinguish representation choice, kernel/model capacity, and augmentation;
- explain what PCA revealed about variance versus task usefulness;
- state at least two limitations or threats to generalization;
- avoid claiming that t-SNE distances, one final test, or this synthetic task prove more than they do.

**Memo:** TODO


## Team contribution, review, sources, and AI-use disclosure

### Contributions and review
For each member, list substantive contributions and at least one part of another member's work that they reviewed.

- **Member 1:** TODO
- **Member 2:** TODO
- **Member 3 (if applicable):** TODO

### Sources / reused code
List any sources or reused code beyond the supplied notebook and course materials. If none, write “None.”

**Response:** TODO

### Generative-AI disclosure
State any permitted AI use (for example, explanation of an error message or brainstorming tests). If none, write “None.” Do not include substantive AI-generated solution material in the submission.

**Response:** TODO

---

### Submission check
Before submitting, restart the kernel and run the completed notebook from top to bottom. Confirm that required tables/plots are visible and that no tuning occurred on the final set before the frozen record.
